# wgnd · Examples
### Alle Funktionen mit `customers.csv`

Dieses Notebook zeigt die aktuelle API von wgnd v0.2.0.
Der Datensatz hat absichtliche Probleme: unschöne Spaltennamen, Nullwerte, Ausreißer, Duplikate.

---

## 0 · Setup

In [ ]:
from wgnd.core.theme import setup
setup()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_raw = pd.read_csv("data/customers.csv")
df_raw.head()

---
## 1 · inspect — EDA

`inspect(df)` ist der Orchestrator. Jede Sub-Funktion kann auch einzeln aufgerufen werden.

### 1.1 · Vollständiger Überblick (Auswahl von Sektionen)

In [ ]:
from wgnd.inspect import inspect

results = inspect(df_raw, sections=["dimensions", "dtypes", "missing", "duplicates"])

### 1.2 · `inspect_dimensions`

In [ ]:
from wgnd.inspect import inspect_dimensions

dim = inspect_dimensions(df_raw)

### 1.3 · `inspect_memory`

In [ ]:
from wgnd.inspect import inspect_memory

mem = inspect_memory(df_raw)

### 1.4 · `inspect_dtypes`

In [ ]:
from wgnd.inspect import inspect_dtypes

dtypes = inspect_dtypes(df_raw)
# Spalten mit Nullwerten herausfiltern:
nulls = dtypes[dtypes["missing_cnt"] > 0]["column"].tolist()
print("Spalten mit Fehlwerten:", nulls)

### 1.5 · `inspect_names`

In [ ]:
from wgnd.inspect import inspect_names

names = inspect_names(df_raw)

### 1.6 · `inspect_missing` — Fehlwert-Heatmap

In [ ]:
from wgnd.inspect import inspect_missing

# chart='heatmap' → Seaborn isnull().T Heatmap (Features auf y-Achse)
# chart=None      → nur Tabelle, kein Chart
missing = inspect_missing(df_raw, chart="heatmap", transpose=True)

### 1.7 · `inspect_duplicates`

In [ ]:
from wgnd.inspect import inspect_duplicates

dupes = inspect_duplicates(df_raw)
# Ergebnis ist ein DataFrame aller duplizierten Zeilen:
dupes.head(6)

### 1.8 · `inspect_numeric_stats`

In [ ]:
from wgnd.inspect import inspect_numeric_stats

# Enthält: missing_pct, mean, median, mean_median_diff (Outlier-Indikator), std, skewness
stats = inspect_numeric_stats(df_raw)

### 1.9 · `inspect_categorical_stats`

In [ ]:
from wgnd.inspect import inspect_categorical_stats

cat_stats = inspect_categorical_stats(df_raw)

### 1.10 · `inspect_correlations` — Heatmap + Target-Ranking + Pairplot

In [ ]:
from wgnd.inspect import inspect_correlations

# Nur Tabellen (keine Charts):
corr = inspect_correlations(df_raw, method="pearson")

# Mit Target-Ranking (welche Features korrelieren am stärksten mit is_churned?):
corr = inspect_correlations(df_raw, target="is_churned")

# Nur Paare über einem Schwellwert:
corr_high = inspect_correlations(df_raw, threshold=0.3)

In [ ]:
# Mit Seaborn-Pairplot (scatter matrix + KDE-Diagonale):
corr = inspect_correlations(df_raw, target="is_churned",
                             show_pairplot=True, pairplot_hue="is_churned")

### 1.11 · `inspect_outliers` — IQR 1.5× und 3×

In [ ]:
from wgnd.inspect import inspect_outliers

outliers = inspect_outliers(df_raw)

### 1.12 · `inspect_outlier_detail` — Boxplot + Histogram pro Feature

In [ ]:
from wgnd.inspect import inspect_outlier_detail

# Boxplot (oben) + Histogram mit KDE (unten), alle IQR-Linien
inspect_outlier_detail(df_raw, "Annual Income (€)", hue="is_churned")

### 1.13 · `iqr_values` — IQR-Grenzen als dict

In [ ]:
from wgnd.inspect import iqr_values

iqr = iqr_values(df_raw, "Annual Income (€)")
print(iqr)

# Typischer Einsatz: Maske für Outlier-Zeilen
mask_outliers = df_raw["Annual Income (€)"] > iqr["upper_1.5x"]
print(f"Outliers oberhalb 1.5× IQR: {mask_outliers.sum()}")

---
## 2 · utils — Helfer

### 2.1 · `memory_report` — Speicher pro Spalte

In [ ]:
from wgnd.utils import memory_report

mem = memory_report(df_raw)

### 2.2 · `downcast` — Speicher reduzieren

In [ ]:
from wgnd.utils import downcast

df_small = downcast(df_raw)
print(df_small.dtypes)

### 2.3 · `timer` — Laufzeit messen

In [ ]:
from wgnd.utils import timer
import time

@timer
def slow_pipeline(df):
    time.sleep(0.3)
    return df.copy()

result = slow_pipeline(df_raw)

---
## 3 · viz — Charts

Alle Funktionen geben `(fig, ax)` zurück und nutzen automatisch das wgnd-Theme.

### 3.0 · Farbpaletten anzeigen

In [ ]:
from wgnd.viz import show_palettes
show_palettes()

### 3.1 · `bar` — Balkendiagramm

In [ ]:
from wgnd.viz import bar

# Einfaches Balkendiagramm
city_avg = df_raw.groupby("city")["avg_order_value"].mean().sort_values()
fig, ax = bar(city_avg, orient="h", title="Avg. Order Value by City",
              ref_val=city_avg.mean(), ref_label="Ø")
plt.show()

### 3.2 · `bar` mit Highlight

In [ ]:
city_avg_sorted = df_raw.groupby("city")["avg_order_value"].mean().sort_values(ascending=False)
fig, ax = bar(city_avg_sorted, orient="v",
              title="Top Cities — Avg. Order Value",
              highlight_top_n=2)
plt.show()

### 3.3 · `stacked_bar` — Gestapelt in Prozent

In [ ]:
from wgnd.viz import stacked_bar

fig, ax = stacked_bar(df_raw, cat_col="segment", target_col="is_churned",
                      normalize=True, title="Churn Rate by Segment")
plt.show()

### 3.4 · `dual_axis` — Volumen + Rate

In [ ]:
from wgnd.viz import dual_axis

stats = df_raw.groupby("city").agg(
    count=("is_churned", "count"),
    churn_rate=("is_churned", "mean")
).reset_index()

fig, ax1, ax2 = dual_axis(stats, x="city", y_bar="count", y_line="churn_rate",
                           ref_val=stats["churn_rate"].mean(),
                           title="Volume vs. Churn Rate by City")
plt.show()

### 3.5 · `line` — Zeitreihe mit Zonen

In [ ]:
from wgnd.viz import line
from wgnd.core.config import cfg

monthly = (df_raw.assign(month=range(len(df_raw)))
           .groupby(df_raw.index // 30)["avg_order_value"].mean())

fig, ax = line(monthly,
    title="Avg. Order Value over Time",
    ylabel="€",
    ref_lines=[{"y": monthly.mean(), "label": "Mean", "style": "signal"}],
    spans=[{"x0": 5, "x1": 8, "label": "Focus Period",
            "color": cfg.PALETTE_CATEGORICAL[0], "alpha": 0.12}])
plt.show()

### 3.6 · `scatter_highlight` — Outlier markieren

In [ ]:
from wgnd.viz import scatter_highlight
from wgnd.inspect import iqr_values

iqr = iqr_values(df_raw, "Annual Income (€)")
mask_out = df_raw["Annual Income (€)"] > iqr["upper_1.5x"]

fig, ax = scatter_highlight(
    df_raw, x="Annual Income (€)", y="avg_order_value",
    highlight_mask=mask_out,
    title="Income vs. Order Value — Outlier Highlighted"
)
plt.show()

### 3.7 · `scatter_highlight` mit Hue

In [ ]:
fig, ax = scatter_highlight(df_raw, x="spend_score", y="avg_order_value",
                             hue="is_churned",
                             title="Spend Score vs. Order Value by Churn")
plt.show()

### 3.8 · `grid_histplot` — Alle Features auf einmal

In [ ]:
from wgnd.viz import grid_histplot

# multiple='fill' → relative Anteile (Risiko-Verlauf)
fig, axes = grid_histplot(df_raw, hue="is_churned", multiple="fill",
                           ref_line=df_raw["is_churned"].mean())
plt.show()

In [ ]:
# multiple='stack' → absolute Mengen
fig, axes = grid_histplot(df_raw, hue="is_churned", multiple="stack")
plt.show()

### 3.9 · `grid_scatter` — Strategische Scatter-Paare

In [ ]:
from wgnd.viz import grid_scatter

combos = [
    ("spend_score", "avg_order_value"),
    ("Annual Income (€)", "avg_order_value"),
    ("n_purchases", "spend_score"),
]

fig, axes = grid_scatter(df_raw, combinations=combos, hue="is_churned",
                          title="Feature Relationships by Churn")
plt.show()

---
## 4 · Eigene Charts mit wgnd-Theme

`mpl_style()` gibt alle kwargs auf einen Schlag zurück.

In [ ]:
from wgnd.core.theme import mpl_style
from wgnd.core.config import cfg
import matplotlib.pyplot as plt
import seaborn as sns

style = mpl_style()

# Beispiel: Seaborn Boxplot nach Gruppe
fig, ax = plt.subplots(figsize=(10, 5))
palette = {0: cfg.PALETTE_CATEGORICAL[0], 1: cfg.COLOR_NEGATIVE}

sns.boxplot(data=df_raw, x="city", y="avg_order_value", hue="is_churned",
            palette=palette, ax=ax, linewidth=0.8)

ax.set_title("Order Value by City & Churn Status", **style["title"])
ax.set_xlabel("City", **style["label"])
ax.set_ylabel("Avg. Order Value", **style["label"])
ax.axhline(df_raw["avg_order_value"].mean(), **style["signal"], label="Mean")
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(cfg.CHART_AXIS)
ax.legend(title="Churned", labels=["No", "Yes", "Mean"])
plt.tight_layout()
plt.show()

In [ ]:
# Beispiel: Doppelachsen (Volumen + Risiko) komplett manuell
import numpy as np

city_stats = df_raw.groupby("city").agg(
    count=("is_churned", "count"),
    churn=("is_churned", "mean")
).sort_values("count", ascending=False).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))

ax1.bar(city_stats["city"], city_stats["count"],
        color=cfg.PALETTE_CATEGORICAL[0], alpha=0.8, width=0.6)
ax1.set_ylabel("Total customers", **style["label"])
ax1.set_title("Volume vs. Churn Risk by City", **style["title"])
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(axis="y", color=cfg.CHART_GRID, linewidth=0.8)

ax2 = ax1.twinx()
ax2.plot(city_stats["city"], city_stats["churn"],
         color=cfg.COLOR_NEGATIVE, linewidth=2, marker="o", markersize=6)
ax2.axhline(city_stats["churn"].mean(), **style["signal"], label="Ø Churn")
ax2.set_ylabel("Churn rate", color=cfg.COLOR_NEGATIVE, fontsize=12, labelpad=8)
ax2.tick_params(axis="y", labelcolor=cfg.COLOR_NEGATIVE)
ax2.spines[["top"]].set_visible(False)

plt.tight_layout()
plt.show()